In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import matplotlib as mpl

# System constants
# Units: Length in AU, Mass in Solar Masses, Time in Years
G = 4 * np.pi**2          # Gravitational constant
M_sun = 1.0               # Mass of the Sun
M_E = 3.003e-6            # Mass of the Earth
M_M = 3.213e-7            # Mass of Mars

R_E = 1.0                 # Earth orbital radius (AU)
R_M = 1.524               # Mars orbital radius (AU)

omega_E = 2 * np.pi       # Earth angular velocity (rad/year)
omega_M = np.sqrt(G * M_sun / R_M**3)  # Mars angular velocity

In [ ]:
def planet_positions(t):
    x_E = R_E * np.cos(omega_E * t)
    y_E = R_E * np.sin(omega_E * t)
    x_M = R_M * np.cos(omega_M * t + phi_0)
    y_M = R_M * np.sin(omega_M * t + phi_0)
    return np.array([x_E, y_E]), np.array([x_M, y_M])

def derivatives(t, Y):
    x, y, vx, vy = Y
    pos_E, pos_M = planet_positions(t)
    x_E, y_E = pos_E
    x_M, y_M = pos_M

    r3_sun = (x**2 + y**2)**1.5
    r3_E = ((x - x_E)**2 + (y - y_E)**2)**1.5
    r3_M = ((x - x_M)**2 + (y - y_M)**2)**1.5

    ax = -G * (M_sun * x / r3_sun + M_E * (x - x_E) / r3_E + M_M * (x - x_M) / r3_M)
    ay = -G * (M_sun * y / r3_sun + M_E * (y - y_E) / r3_E + M_M * (y - y_M) / r3_M)

    return np.array([vx, vy, ax, ay])

def rk4_step(t, Y, h):
    k1 = h * derivatives(t, Y)
    k2 = h * derivatives(t + h/2, Y + k1/2)
    k3 = h * derivatives(t + h/2, Y + k2/2)
    k4 = h * derivatives(t + h, Y + k3)
    return Y + (1.0/6.0) * (k1 + 2*k2 + 2*k3 + k4)

In [ ]:
# Simulation parameters
t_final = 1.25   # Total simulation time (1.25 years)
h = 0.0001       # Integration step size

# First burn calculation (Trans-Mars Injection)
a_transfer = (R_E + R_M) / 2.0
v_earth_orbit = omega_E * R_E
v_transfer_peri = np.sqrt(G * M_sun * (2.0 / R_E - 1.0 / a_transfer))
v_infty = v_transfer_peri - v_earth_orbit

# Place spacecraft at the edge of Earth's sphere of influence
earth_offset = 0.005
x0 = R_E + earth_offset
y0 = 0.0
vx0 = 0.0

# Add escape velocity to leave Earth
v_esc_offset = np.sqrt(2.0 * G * M_E / earth_offset)
v_rel_needed = np.sqrt(v_infty**2 + v_esc_offset**2)

vy0 = v_earth_orbit + v_rel_needed
delta_v1 = vy0 - v_earth_orbit

v_conv = 149597870.7 / 31557600  # Conversion factor from AU/year to km/s
delta_v1_kms = delta_v1 * v_conv

print(f"Initial conditions set. Delta V1 (Injection): {delta_v1_kms:.2f} km/s")

Y0 = np.array([x0, y0, vx0, vy0])

In [ ]:
# Time array definition
times = np.arange(0, t_final, h)
num_steps = len(times)

# Phase angles to compare
phi_values = [0.60, 0.771, 0.79, 1.00]

distance_results = {}
spacecraft_results = {}

print("Starting transfer simulations...")

for phi_option in phi_values:
    phi_0 = phi_option

    Y = Y0.copy()
    reached_mars = False
    capture_success = False
    prev_distance = float('inf')

    offset_x, offset_y = 0.0, 0.0
    delta_v2_kms = 0.0
    distance_history = np.zeros(num_steps)
    spacecraft_history = np.zeros((num_steps, 2))

    arrival_time = 0.0
    min_recorded_dist = float('inf')

    for i in range(num_steps):
        t = times[i]
        pE, pM = planet_positions(t)

        dist_mars = np.linalg.norm([Y[0] - pM[0], Y[1] - pM[1]])
        distance_history[i] = dist_mars

        if not reached_mars and t > 0.1:
            if dist_mars > prev_distance:
                reached_mars = True
                arrival_time = t
                min_recorded_dist = prev_distance

                if prev_distance < 0.05:
                    capture_success = True
                    offset_x = Y[0] - pM[0]
                    offset_y = Y[1] - pM[1]

                    # Braking burn calculation (Delta V2)
                    v_M_x = -omega_M * pM[1]
                    v_M_y =  omega_M * pM[0]
                    dv_x = v_M_x - Y[2]
                    dv_y = v_M_y - Y[3]
                    delta_v2_kms = np.sqrt(dv_x**2 + dv_y**2) * v_conv

        prev_distance = dist_mars

        if capture_success:
            Y[0] = pM[0] + offset_x
            Y[1] = pM[1] + offset_y
            Y[2] = -omega_M * pM[1]
            Y[3] =  omega_M * pM[0]
        else:
            Y = rk4_step(t, Y, h)

        spacecraft_history[i] = [Y[0], Y[1]]

    distance_results[phi_option] = distance_history
    spacecraft_results[phi_option] = spacecraft_history

    # Print results to terminal
    print(f"\n--- Results for phi_0 = {phi_option:.3f} rad ---")
    if reached_mars:
        flight_days = arrival_time * 365.25
        print(f"Time of flight:     {arrival_time:.3f} years ({flight_days:.0f} days)")
        print(f"Minimum distance:   {min_recorded_dist:.4f} AU")
        if capture_success:
            print("Final state:        Capture (escort regime)")
            print(f"Injection burn:     Delta V1 = {delta_v1_kms:.2f} km/s")
            print(f"Insertion burn:     Delta V2 = {delta_v2_kms:.2f} km/s")
            print(f"Total cost:         Delta V  = {delta_v1_kms + delta_v2_kms:.2f} km/s")
        else:
            print("Final state:        Flyby (distance too large for capture)")
    else:
        print("Final state:        Lost (no Mars approach)")

print("Simulation finished")

In [ ]:
phi_animate = 0.79

# Plot setup for animation
phi_0 = phi_animate
spacecraft_anim_data = spacecraft_results[phi_animate]

plt.style.use('dark_background')
mpl.rcParams['animation.embed_limit'] = 100

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect('equal')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.grid(False)

ax.set_title(rf'Mission to Mars ($\phi_0$ = {phi_animate:.3f} rad)', color='white', pad=15)
ax.set_xlabel('x (AU)', color='white')
ax.set_ylabel('y (AU)', color='white')

ax.plot(0, 0, 'o', color='gold', markersize=15, label='Sun')

bg_theta = np.linspace(0, 2*np.pi, 200)
ax.plot(R_E*np.cos(bg_theta), R_E*np.sin(bg_theta), '--', color='dodgerblue', alpha=0.3)
ax.plot(R_M*np.cos(bg_theta), R_M*np.sin(bg_theta), '--', color='tomato', alpha=0.3)

spacecraft_line, = ax.plot([], [], '-', color='cyan', linewidth=2)
spacecraft_dot, = ax.plot([], [], 'o', color='white', markersize=3, zorder=3, label='Spacecraft')
earth_dot, = ax.plot([], [], 'o', color='dodgerblue', markersize=9, label='Earth')
mars_dot, = ax.plot([], [], 'o', color='tomato', markersize=7, label='Mars', zorder=2)

legend = ax.legend(loc='upper right', facecolor='black', edgecolor='white')
for text in legend.get_texts(): text.set_color("white")

def init():
    spacecraft_line.set_data([], [])
    return spacecraft_line, spacecraft_dot, earth_dot, mars_dot

frame_skip = 30

def update(frame):
    idx = frame * frame_skip
    t = times[idx]

    pE, pM = planet_positions(t)

    spacecraft_line.set_data(spacecraft_anim_data[:idx, 0], spacecraft_anim_data[:idx, 1])
    spacecraft_dot.set_data([spacecraft_anim_data[idx, 0]], [spacecraft_anim_data[idx, 1]])
    earth_dot.set_data([pE[0]], [pE[1]])
    mars_dot.set_data([pM[0]], [pM[1]])

    return spacecraft_line, spacecraft_dot, earth_dot, mars_dot

total_frames = len(times) // frame_skip
ani = animation.FuncAnimation(fig, update, frames=total_frames, init_func=init, blit=True, interval=20)

gif_name = f'mars_mission_phi_{phi_animate:.3f}.gif'
ani.save(gif_name, writer='pillow', fps=40)
print(f"GIF saved successfully: {gif_name}")

plt.close()
display(HTML(ani.to_jshtml()))

In [ ]:
plt.style.use('default')
fig, ax = plt.subplots(figsize=(8, 5.5))

colors = ['tab:red', 'tab:orange', 'tab:green', 'tab:blue']
styles = ['--', ':', '-', '-.']

for phi_option, col, st in zip(phi_values, colors, styles):
    ax.plot(times, distance_results[phi_option], color=col, linestyle=st,
            linewidth=2.0, label=rf'$\phi_0$ = {phi_option:.3f} rad')

ax.set_title('Distance Evolution by Launch Window', fontsize=16, pad=15)
ax.set_xlabel('Time of flight (years)', fontsize=14)
ax.set_ylabel('Spacecraft - Mars Distance (AU)', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(loc='upper right', frameon=True, shadow=False, fontsize=13)

plt.tight_layout()
plt.savefig('launch_windows_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Sensitivity analysis: Launch windows
test_angles = np.linspace(0.4, 1.2, 400)
min_distances = []

print("Computing periapsis distances for phase sweep...")

for phi_test in test_angles:
    phi_0_temp = phi_test
    Y_test = Y0.copy()
    dist_min_test = float('inf')

    for t_test in np.arange(0, 0.8, 0.005):
        # Manual Mars position calculation for current phase
        pM_x = R_M * np.cos(omega_M * t_test + phi_0_temp)
        pM_y = R_M * np.sin(omega_M * t_test + phi_0_temp)

        d_m = np.linalg.norm([Y_test[0] - pM_x, Y_test[1] - pM_y])
        if d_m < dist_min_test:
            dist_min_test = d_m

        Y_test = rk4_step(t_test, Y_test, 0.005)

    min_distances.append(dist_min_test)

# Sweep plot
plt.style.use('default')
fig, ax = plt.subplots(figsize=(9, 6))

# Main plot
ax.plot(test_angles, min_distances, 'b-', linewidth=2)
ax.axvline(x=0.771, color='r', linestyle='--', label=r'Theoretical optimum ($\phi_0 \approx 0.771$ rad)')

ax.set_title('Mission Sensitivity to Launch Window', fontsize=16, pad=15)
ax.set_xlabel('Initial Mars phase angle (radians)', fontsize=14)
ax.set_ylabel('Minimum approach distance (AU)', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.set_yscale('log')
ax.grid(True, alpha=0.5)
ax.set_ylim(bottom=0.0001)
ax.legend(loc='upper right', fontsize=12)

# Inset (Zoom into capture zone)
# [rel_x, rel_y, rel_width, rel_height]
axins = ax.inset_axes([0.12, 0.12, 0.35, 0.35])
axins.plot(test_angles, min_distances, 'b-', linewidth=2)
axins.axvline(x=0.771, color='r', linestyle='--')

axins.set_xlim(0.74, 0.82)
axins.set_ylim(0.0003, 0.05)
axins.set_yscale('log')
axins.grid(True, alpha=0.3)
axins.set_title('Capture zone zoom', fontweight='bold')
axins.tick_params(axis='both', which='major', labelsize=10)

ax.indicate_inset_zoom(axins, edgecolor="black")

plt.tight_layout()
plt.savefig('initial_phase_sweep_inset.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Static simulation frames
phi_static = 0.6  # Change this value to generate phase diagrams for other windows
phi_0 = phi_static
spacecraft_static_data = spacecraft_results[phi_static]

key_times = [0.0, 0.32, 0.643, 1.0]
filenames = ['phase_1_injection06.png', 'phase_2_transit06.png', 'phase_3_capture06.png', 'phase_4_escort06.png']
titles = ['Phase 1: Trans-Mars Injection',
          'Phase 2: Inertial Coasting',
          'Phase 3: Orbital Rendezvous Maneuver',
          'Phase 4: Stationary Escort Regime']

plt.style.use('dark_background')

for t_target, filename, title in zip(key_times, filenames, titles):
    idx = (np.abs(times - t_target)).argmin()
    t_exact = times[idx]
    pE, pM = planet_positions(t_exact)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.grid(False)

    ax.set_title(f"{title} (t = {t_exact:.3f} years)", color='white', fontsize=12, pad=10)
    ax.set_xlabel('x (AU)', color='white')
    ax.set_ylabel('y (AU)', color='white')

    bg_theta = np.linspace(0, 2*np.pi, 200)
    ax.plot(R_E*np.cos(bg_theta), R_E*np.sin(bg_theta), '--', color='dodgerblue', alpha=0.3, zorder=1)
    ax.plot(R_M*np.cos(bg_theta), R_M*np.sin(bg_theta), '--', color='tomato', alpha=0.3, zorder=1)

    ax.plot(0, 0, 'o', color='gold', markersize=15, label='Sun', zorder=2)
    ax.plot(pE[0], pE[1], 'o', color='dodgerblue', markersize=9, label='Earth', zorder=5)
    ax.plot(pM[0], pM[1], 'o', color='tomato', markersize=7, label='Mars', zorder=3)

    ax.plot(spacecraft_static_data[:idx+1, 0], spacecraft_static_data[:idx+1, 1], '-', color='cyan', linewidth=2, zorder=4)
    ax.plot(spacecraft_static_data[idx, 0], spacecraft_static_data[idx, 1], 'o', color='white', markersize=3, zorder=5, label='Spacecraft')

    ax.legend(loc='upper right', facecolor='black', edgecolor='white', fontsize=9)

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

print(f"Phase diagrams rendered for phi_0 = {phi_static:.3f}")

In [ ]:
# Energy cost analysis within the viable capture window (< 0.05 AU)
capture_angles = np.linspace(0.7, 0.82, 80)
valid_angles = []
delta_v2_list = []
delta_vtot_list = []
flight_time_list = []

print("Computing energies...")

for phi_test in capture_angles:
    phi_0 = phi_test
    Y = Y0.copy()
    prev_distance = float('inf')
    reached_mars = False

    for t in times:
        pE, pM = planet_positions(t)
        dist_mars = np.linalg.norm([Y[0] - pM[0], Y[1] - pM[1]])

        if not reached_mars and t > 0.1:
            if dist_mars > prev_distance:
                reached_mars = True

                if prev_distance < 0.05:
                    v_M_x = -omega_M * pM[1]
                    v_M_y =  omega_M * pM[0]
                    dv_x = v_M_x - Y[2]
                    dv_y = v_M_y - Y[3]
                    dv2 = np.sqrt(dv_x**2 + dv_y**2) * v_conv

                    valid_angles.append(phi_test)
                    delta_v2_list.append(dv2)
                    delta_vtot_list.append(delta_v1_kms + dv2)
                    flight_time_list.append(t * 365.25)
                break

        prev_distance = dist_mars
        Y = rk4_step(t, Y, h)

# Dual-axis plot
plt.style.use('default')
fig, ax1 = plt.subplots(figsize=(9, 6))

color1 = 'red'
color2 = 'blue'

ax1.set_xlabel('Initial Mars phase angle (radians)', fontsize=14)

ax1.set_ylabel(r'Total Mission Cost, $\Delta V_{tot}$ (km/s)', color=color1, fontsize=14, y=0.5)
line1 = ax1.plot(valid_angles, delta_vtot_list, color=color1, linewidth=2.5, marker='o', markersize=4, label=r'$\Delta V_{tot}$')
ax1.tick_params(axis='y', labelcolor=color1, labelsize=12)
ax1.tick_params(axis='x', labelsize=12)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.set_ylabel('Time of flight (Earth days)', color=color2, fontsize=14)
line2 = ax2.plot(valid_angles, flight_time_list, color=color2, linewidth=2.5, linestyle='--', marker='s', markersize=4, label='Transit time')
ax2.tick_params(axis='y', labelcolor=color2, labelsize=12)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=12, framealpha=0.9)

# Optimum annotation
opt_idx = np.argmin(delta_vtot_list)
ax1.axvline(x=valid_angles[opt_idx], color='gray', linestyle=':', alpha=0.8)
ax1.annotate(fr'Energy optimum' + '\n' + fr'$\phi_0 \approx$ {valid_angles[opt_idx]:.3f} rad' + '\n' + fr'$\Delta V \approx$ {delta_vtot_list[opt_idx]:.2f} km/s',
             xy=(valid_angles[opt_idx], delta_vtot_list[opt_idx]),
             xytext=(valid_angles[opt_idx] - 0.015, delta_vtot_list[opt_idx] + 0.04),
             arrowprops=dict(facecolor='black', arrowstyle='->'),
             fontsize=12, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9))

plt.title('Energy Cost vs. Time of Flight Trade-off', fontsize=16, pad=30)
fig.tight_layout()
plt.savefig('energy_cost_analysis.png', dpi=300)
plt.show()